<a href="https://colab.research.google.com/github/Arfa-Tariq/AstroPlanner-AI/blob/main/notebooks/01_user_observation_request.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#AstroPlanner AI: Intelligent Observation and Astrophotography Planning Assistant
## Date: 30 June 2026


--------------------------------------------------------------------------
## Module 1: User Observation Request

### Objective : Design and implement the input layer of the application.




### Understanding User Requirements & Identifying Required Inputs
#### Collect user information including:
-	Geographic location
-	Experience level
-	Telescope specifications
-	Camera specifications
-	Mount details

#### Types of Users:

- Beginner
- Hobbyist
- Astrophotographer
- Experienced observer


### Step 1 — Define the data model first, before any input logic

In [1]:
!pip install pydantic -q

In [12]:
import re
from datetime import date as date_type, timedelta
from enum import Enum
from typing import Optional

from pydantic import BaseModel, Field, field_validator

#### Models

In [13]:
from pydantic import BaseModel, Field
from typing import Optional
from enum import Enum
from datetime import date


class ExperienceLevel(str, Enum):
    """
    Self-reported astronomy experience level of the user.

    Used downstream by the Recommendation Engine to adjust target difficulty
    (e.g. beginners get bright, easy Messier objects; advanced users get
    faint NGC objects or narrowband targets) and to tune how much explanation
    the LLM assistant includes in its responses.
    """
    beginner = "beginner"
    intermediate = "intermediate"
    advanced = "advanced"


class TelescopeSpec(BaseModel):
    """
    Optical specifications of the user's telescope.

    These values drive two downstream calculations:
    1. Field-of-View Analyzer — focal_length_mm combined with the camera's
       sensor dimensions determines the imaging FoV in arcminutes.
    2. Limiting magnitude / resolving power estimates — aperture_mm determines
       how faint an object the telescope can usefully observe, which feeds
       into the Recommendation Engine's target ranking.
    """
    aperture_mm: float = Field(
        ...,
        description=(
            "Diameter of the telescope's primary lens or mirror, in millimeters. "
            "Determines light-gathering power and resolving ability; larger "
            "apertures reveal fainter objects and finer detail."
        ),
        gt=0,
    )
    focal_length_mm: float = Field(
        ...,
        description=(
            "Distance from the primary optic to the focal point, in millimeters. "
            "Used together with camera sensor size to compute imaging field of view, "
            "and together with aperture to derive focal_ratio."
        ),
        gt=0,
    )
    focal_ratio: Optional[float] = Field(
        default=None,
        description=(
            "Focal ratio (f/number) of the optical system, calculated as "
            "focal_length_mm / aperture_mm if not provided directly. Lower "
            "values (e.g. f/4) are 'faster' and better suited to wide-field "
            "deep-sky imaging; higher values (e.g. f/10) are 'slower' and "
            "better suited to high-magnification planetary/lunar imaging."
        ),
    )


class CameraSpec(BaseModel):
    """
    Imaging sensor specifications for the user's astrophotography camera.

    Sensor dimensions and pixel size are required inputs for the Field-of-View
    Analyzer (Phase 5), which determines whether a given celestial target will
    fit perfectly, be too large, or be too small in a single frame.
    """
    sensor_width_mm: float = Field(
        ...,
        description="Physical width of the camera's image sensor, in millimeters.",
        gt=0,
    )
    sensor_height_mm: float = Field(
        ...,
        description="Physical height of the camera's image sensor, in millimeters.",
        gt=0,
    )
    pixel_size_um: float = Field(
        ...,
        description=(
            "Size of a single pixel on the sensor, in micrometers. Used to "
            "calculate image scale (arcseconds per pixel) together with the "
            "telescope's focal length, which affects whether the setup is "
            "well-matched ('oversampled' or 'undersampled') for a given target."
        ),
        gt=0,
    )


class MountSpec(BaseModel):
    """
    Specifications of the telescope mount, which governs tracking accuracy
    and how easily the user can locate and follow targets across the sky.

    goto_capable in particular affects the Recommendation Engine: non-GoTo
    mounts may warrant simpler, easier-to-locate targets for beginners.
    """
    type: str = Field(
        ...,
        description=(
            "Mount type, e.g. 'alt-az' or 'equatorial'. Equatorial mounts can "
            "track the sky's rotation along a single axis, which matters for "
            "long-exposure astrophotography; alt-az mounts are simpler but "
            "introduce field rotation during long exposures."
        ),
    )
    goto_capable: bool = Field(
        ...,
        description=(
            "Whether the mount can automatically slew to and track a chosen "
            "target. Affects how detailed the Observation Scheduler's pointing "
            "instructions need to be."
        ),
    )


class UserProfile(BaseModel):
    """
    Complete profile of a single user, combining their location, experience
    level, and equipment. This is the core identity object that gets reused
    across every downstream module: Weather Intelligence (uses location),
    Celestial Visibility Engine (uses location), Recommendation Engine (uses
    experience_level + equipment), and Field-of-View Analyzer (uses telescope
    + camera).
    """
    name: str = Field(..., description="Display name of the user.")
    latitude: float = Field(
        ...,
        description=(
            "Observing site latitude in decimal degrees (positive = North, "
            "negative = South). Required for computing object rise/set times, "
            "altitude/azimuth, and for fetching local weather data."
        ),
        ge=-90,
        le=90,
    )
    longitude: float = Field(
        ...,
        description=(
            "Observing site longitude in decimal degrees (positive = East, "
            "negative = West). Required alongside latitude for all astronomical "
            "and weather calculations."
        ),
        ge=-180,
        le=180,
    )
    experience_level: ExperienceLevel = Field(
        ...,
        description="See ExperienceLevel; used to tailor target difficulty and explanation depth.",
    )
    telescope: TelescopeSpec = Field(
        ..., description="The user's primary telescope specifications."
    )
    camera: Optional[CameraSpec] = Field(
        default=None,
        description=(
            "The user's astrophotography camera, if any. Optional because "
            "visual-only observers (no imaging) won't have one; in that case "
            "Field-of-View analysis is skipped."
        ),
    )
    mount: Optional[MountSpec] = Field(
        default=None,
        description="The user's mount specifications, if known.",
    )


class ObservationRequest(BaseModel):
    """
    Top-level request object representing a single observation-planning
    session. This is the single input object that gets passed into every
    downstream notebook/module (weather lookup, visibility engine,
    recommendation engine), and later becomes the structured input/output
    schema referenced by the LangChain tools and the LLM assistant.
    """
    user: UserProfile = Field(..., description="The requesting user's full profile.")
    observation_date: date = Field(
        ...,
        description=(
            "The calendar date of the planned observing session (local to the "
            "user's site). Used to fetch the relevant weather forecast and to "
            "compute that night's specific rise/set/altitude data."
        ),
    )
    notes: Optional[str] = Field(
        default=None,
        description=(
            "Freeform notes from the user, e.g. 'want to image a galaxy' or "
            "'first time using this telescope'. Not used in calculations directly, "
            "but can be passed to the LLM assistant as conversational context."
        ),
    )

### Step 2 — a quick test instantiation

In [14]:
test_request = ObservationRequest(
    user=UserProfile(
        name="Ali",
        latitude=33.6844,
        longitude=73.0479,
        experience_level="beginner",
        telescope=TelescopeSpec(aperture_mm=150, focal_length_mm=750),
        camera=CameraSpec(sensor_width_mm=23.5, sensor_height_mm=15.6, pixel_size_um=3.76),
        mount=MountSpec(type="equatorial", goto_capable=True),
    ),
    observation_date=date(2026, 7, 5),
)
print(test_request.model_dump_json(indent=2))

{
  "user": {
    "name": "Ali",
    "latitude": 33.6844,
    "longitude": 73.0479,
    "experience_level": "beginner",
    "telescope": {
      "aperture_mm": 150.0,
      "focal_length_mm": 750.0,
      "focal_ratio": null
    },
    "camera": {
      "sensor_width_mm": 23.5,
      "sensor_height_mm": 15.6,
      "pixel_size_um": 3.76
    },
    "mount": {
      "type": "equatorial",
      "goto_capable": true
    }
  },
  "observation_date": "2026-07-05",
  "notes": null
}


#### Input validation helpers

In [18]:
def prompt_float(label: str, min_val: float = None, max_val: float = None) -> float:
    """Repeatedly prompts until a valid float within optional bounds is entered."""
    while True:
        raw = input(f"{label}: ").strip()
        try:
            value = float(raw)
        except ValueError:
            print("  Invalid number, try again.")
            continue
        if min_val is not None and value < min_val:
            print(f"  Must be greater than {min_val}.")
            continue
        if max_val is not None and value > max_val:
            print(f"  Must be at most {max_val}.")
            continue
        return value


def prompt_choice(label: str, choices: list[str]) -> str:
    """Repeatedly prompts until input matches one of the allowed choices (case-insensitive)."""
    choices_lower = [c.lower() for c in choices]
    while True:
        raw = input(f"{label} [{'/'.join(choices)}]: ").strip().lower()
        if raw in choices_lower:
            return raw
        print(f"  Must be one of: {', '.join(choices)}.")


def prompt_yes_no(label: str) -> bool:
    """Repeatedly prompts until a y/n answer is given."""
    while True:
        raw = input(f"{label} [y/n]: ").strip().lower()
        if raw in ("y", "yes"):
            return True
        if raw in ("n", "no"):
            return False
        print("  Please enter y or n.")


def sanitize_free_text(raw: str, max_len: int = 300) -> str:
    """
    Sanitizes freeform text fields that may later be passed to an LLM agent
    as context (e.g. 'notes', fed to the assistant in notebook 06). Strips
    control characters and enforces a length cap to reduce prompt-injection
    surface and prevent context/token-eating paste-bombs. This is reserved
    for genuinely freeform fields — structured-looking fields (name, choices,
    numbers) should use a dedicated format-specific validator instead.
    """
    cleaned = re.sub(r"[\x00-\x1f\x7f]", "", raw).strip()
    return cleaned[:max_len]


def prompt_name(label: str = "Name", max_len: int = 100) -> str:
    """
    Prompts for a person's name, retrying until input contains only letters,
    spaces, hyphens, and apostrophes (covers names like 'Anne-Marie' or
    'O'Brien'), with no digits or symbols.
    """
    name_pattern = re.compile(r"^[A-Za-z][A-Za-z\s\-']{1,99}$")
    while True:
        raw = input(f"{label}: ")
        cleaned = sanitize_free_text(raw, max_len=max_len)
        if name_pattern.match(cleaned):
            return cleaned
        print("  Name must contain only letters, spaces, hyphens, or apostrophes (no numbers/symbols).")


def prompt_observation_date(max_days_ahead: int = 7) -> str:
    """
    Prompts for the planned observation date, defaulting to today if the
    user presses Enter. Capped to a 7-day forward window to match weather
    forecast reliability and the system's 'best night this week' use case.
    """
    today = date_type.today()
    max_date = today + timedelta(days=max_days_ahead)

    while True:
        raw = input(
            f"Observation date (YYYY-MM-DD, press Enter for tonight [{today.isoformat()}], "
            f"up to {max_date.isoformat()}): "
        ).strip()

        if not raw:
            return today.isoformat()

        try:
            parsed = date_type.fromisoformat(raw)
        except ValueError:
            print("  Invalid date format, use YYYY-MM-DD.")
            continue

        if parsed < today:
            print("  Date cannot be in the past.")
            continue
        if parsed > max_date:
            print(f"  Date cannot be more than {max_days_ahead} days ahead.")
            continue

        return parsed.isoformat()

#### Main collector

In [19]:
def collect_observation_request_cli() -> ObservationRequest:
    """
    Collects observation request data via terminal prompts, validating and
    retrying on bad input at the source, then returns a Pydantic-validated
    ObservationRequest. This is intentionally the ONLY function that depends
    on the input method (CLI here) — everything downstream consumes
    ObservationRequest objects regardless of how they were built, so this
    function can later be swapped for a web form without touching anything
    else in the system.
    """
    print("=== AstroPlanner: New Observation Request ===")

    raw = {
        "user": {
            "name": prompt_name(),
            "latitude": prompt_float("Latitude (decimal degrees)", -90, 90),
            "longitude": prompt_float("Longitude (decimal degrees)", -180, 180),
            "experience_level": prompt_choice(
                "Experience level", ["beginner", "intermediate", "advanced"]
            ),
            "telescope": {
                "aperture_mm": prompt_float("Telescope aperture (mm)", 10, 1000),
                "focal_length_mm": prompt_float("Telescope focal length (mm)", 100, 5000),
            },
        },
        "observation_date": prompt_observation_date(),
    }

    if prompt_yes_no("Do you have a camera for imaging?"):
        raw["user"]["camera"] = {
            "sensor_width_mm": prompt_float("Sensor width (mm)", 4, 50),
            "sensor_height_mm": prompt_float("Sensor height (mm)", 4, 50),
            "pixel_size_um": prompt_float("Pixel size (µm)", 1, 15),
        }

    if prompt_yes_no("Do you know your mount type?"):
        raw["user"]["mount"] = {
            "type": prompt_choice("Mount type", ["alt-az", "equatorial"]),
            "goto_capable": prompt_yes_no("GoTo capable?"),
        }

    notes_raw = input("Notes (optional, press Enter to skip): ")
    if notes_raw.strip():
        raw["notes"] = sanitize_free_text(notes_raw)

    return ObservationRequest(**raw)

In [20]:
request = collect_observation_request_cli()
print("\n=== Validated Observation Request ===")
print(request.model_dump_json(indent=2))

=== AstroPlanner: New Observation Request ===
Name: -2sda
  Name must contain only letters, spaces, hyphens, or apostrophes (no numbers/symbols).
Name: Arfa
Latitude (decimal degrees): 10000000000000
  Must be at most 90.
Latitude (decimal degrees): 99
  Must be at most 90.
Latitude (decimal degrees): 90.001
  Must be at most 90.
Latitude (decimal degrees): 90
Longitude (decimal degrees): 1000000000000000000
  Must be at most 180.
Longitude (decimal degrees): -10
Experience level [beginner/intermediate/advanced]: beginner
Telescope aperture (mm): 10000000000000000000
  Must be at most 1000.
Telescope aperture (mm): 0
  Must be greater than 10.
Telescope aperture (mm): 10000000000000
  Must be at most 1000.
Telescope aperture (mm): 0000
  Must be greater than 10.
Telescope aperture (mm): 1000
Telescope focal length (mm): 22222222222222222
  Must be at most 5000.
Telescope focal length (mm): 123
Observation date (YYYY-MM-DD, press Enter for tonight [2026-06-30], up to 2026-07-07): 
Do yo